# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Qwen3-VL-4B-Instruct QLoRA fine-tuning for road damage detection (E2, E3, E4).


In [ ]:
# 1. Environment & Dependencies
import os, sys, pathlib, subprocess, shutil, json, time, random, gc
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/input").exists()
print("Running on Kaggle:", IS_KAGGLE)
!nvidia-smi -L

# Use /tmp to avoid cluttering /kaggle/working with 2000 images
BASE = Path("/tmp" if IS_KAGGLE else "/content")
BASE.mkdir(parents=True, exist_ok=True)
REPO = BASE / "K4-Track-4-Day10-VisionLanguageModel"

if not (REPO / "scripts" / "prepare_data.py").exists():
    subprocess.run(["git", "clone", "https://github.com/TheViet298/K4-Track-4-Day10-VisionLanguageModel.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)

os.chdir(str(REPO))
print("Working directory:", os.getcwd())

!pip -q install -r requirements-colab.txt
!pip -q install --upgrade --no-cache-dir "bitsandbytes>=0.46.1"
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)


In [ ]:
# 2. Data Preparation
attached = list(Path("/kaggle/input").rglob("labels-YOLO"))
if attached:
    dataset_source = str(attached[0].parent)
    print("Using attached dataset:", dataset_source)
    subprocess.run(["python", "scripts/prepare_data.py", "--source", dataset_source], check=True)
else:
    subprocess.run(["python", "scripts/prepare_data.py", "--kaggle"], check=True)

assert Path("data/splits/meta.json").exists(), "meta.json missing!"
print("Dataset prepared successfully.")


In [ ]:
# 3. Helpers & Chat Template
import collections
import torch
from transformers import AutoProcessor, BitsAndBytesConfig
try:
    from transformers import Qwen3VLForConditionalGeneration
except ImportError:
    from transformers import Qwen2_5_VLForConditionalGeneration as Qwen3VLForConditionalGeneration
from peft import LoraConfig, get_peft_model
from torch.nn.utils import clip_grad_norm_

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl("data/splits/train.jsonl")
val_rows = read_jsonl("data/splits/val.jsonl")
test_rows = read_jsonl("data/splits/test.jsonl")
print(f"Loaded: train={len(train_rows)}, val={len(val_rows)}, test={len(test_rows)}")

def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    vals = [(xc-w/2)*1000, (yc-h/2)*1000, (xc+w/2)*1000, (yc+h/2)*1000]
    return {"label": label, "bbox_2d": [max(0, min(1000, round(v))) for v in vals]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
PROMPT = ("Detect road damage in this image. Return ONLY a JSON list. Classes are pothole, crack, manhole. " +
          "Every bbox_2d coordinate must be an integer on the 0-1000 normalized scale. " +
          'Format: [{"label":"pothole","bbox_2d":[x1,y1,x2,y2]}].')

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

def make_messages(row, answer=None):
    path = row["image_path"] if os.path.exists(row["image_path"]) else os.path.join("data", row["image_path"])
    image = Image.open(path).convert("RGB")
    user = {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPT}]}
    if answer is None:
        return [user]
    return [user, {"role": "assistant", "content": [{"type": "text", "text": answer}]}]

def encode_example(row):
    answer = objects_to_answer(row["objects"])
    prompt = processor.apply_chat_template(make_messages(row), tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
    full = processor.apply_chat_template(make_messages(row, answer), tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt")
    n = prompt["input_ids"].shape[1]
    labels = full["input_ids"].clone()
    labels[:, :n] = -100
    full["labels"] = labels
    return full

class_rows = {c: [r for r in train_rows if any(o["label"] == c for o in r["objects"])] for c in ("pothole", "crack", "manhole")}
print("Class counts in train:", {k: len(v) for k, v in class_rows.items()})


In [ ]:
# 4. Training & Scoring Engine with True Float32 Projector Protection
def apply_setup(model, setup, rank=16):
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()
    targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    if setup == "E3":
        targets += ["qkv", "proj", "linear_fc1", "linear_fc2"]
    cfg = LoraConfig(r=rank, lora_alpha=2*rank, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=targets)
    peft_model = get_peft_model(model, cfg)
    # Strictly freeze all non-trainable weights
    for p in peft_model.parameters():
        p.requires_grad = False
    for name, p in peft_model.named_parameters():
        if "lora_" in name:
            p.requires_grad = True
    # In E2/E3, unfreeze visual merger in float32 and shield from FP16 downcast
    if setup in ("E2", "E3"):
        for name, p in peft_model.named_parameters():
            if "visual.merger." in name or "visual.deepstack_merger_list." in name:
                p.data = p.data.float()
                p.requires_grad = True
        # Wrap projector forward to execute safely in float32 and clamp
        merger_mods = []
        for name, mod in peft_model.named_modules():
            if name.endswith("visual.merger") or ("visual.deepstack_merger_list" in name and len(name.split(".")) == 4):
                merger_mods.append(mod)
        for mod in merger_mods:
            orig_fwd = mod.forward
            def _make_safe(orig):
                def _safe_forward(*args, **kwargs):
                    with torch.autocast("cuda", enabled=False):
                        args = [a.float() if isinstance(a, torch.Tensor) and a.is_floating_point() else a for a in args]
                        kwargs = {k: (v.float() if isinstance(v, torch.Tensor) and v.is_floating_point() else v) for k, v in kwargs.items()}
                        res = orig(*args, **kwargs)
                        if isinstance(res, torch.Tensor):
                            res = torch.clamp(res, -65000.0, 65000.0).to(torch.float16)
                        return res
                return _safe_forward
            mod.forward = _make_safe(orig_fwd)
    peft_model.print_trainable_parameters()
    return peft_model

def run_experiment(setup_name, rank=16, base_lr=1e-5, steps=300, grad_accum=4):
    eval_name = setup_name.lower()
    evidence_path = Path(f"report/evidence/{eval_name}.json")
    if evidence_path.exists():
        print(f"Evidence {evidence_path} already exists! Skipping.")
        return

    print(f"\n=======================================================")
    print(f" RUNNING EXPERIMENT: {setup_name} (rank={rank})")
    print(f"=======================================================")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quant,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    peft_model = apply_setup(base_model, setup_name, rank=rank)
    peft_model.config.use_cache = False
    try:
        peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        peft_model.gradient_checkpointing_enable()

    trainable = [p for p in peft_model.parameters() if p.requires_grad]
    print(f"[{setup_name}] Trainable tensors: {len(trainable)}, Total params: {sum(p.numel() for p in trainable):,}")
    opt = torch.optim.AdamW(trainable, lr=base_lr, weight_decay=0.0)

    start_time = time.time()
    peft_model.train()
    opt.zero_grad(set_to_none=True)
    rng = random.Random(42)
    warmup_steps = 20

    for step in range(steps):
        if step < warmup_steps:
            curr_lr = base_lr * (step + 1) / warmup_steps
            for pg in opt.param_groups:
                pg["lr"] = curr_lr

        losses = []
        for micro in range(grad_accum):
            cls = ("pothole", "crack", "manhole")[(step * grad_accum + micro) % 3]
            row = rng.choice(class_rows[cls])
            batch = encode_example(row)
            batch = {k: v.to(peft_model.device) if hasattr(v, "to") else v for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = peft_model(**batch).loss / grad_accum
            if not torch.isfinite(loss):
                print(f"Warning: non-finite loss at step {step}, micro {micro}. Skipping.")
                continue
            loss.backward()
            losses.append(loss.item() * grad_accum)

        bad_grad = any(p.grad is not None and not torch.isfinite(p.grad).all() for p in trainable)
        if bad_grad or len(losses) == 0:
            print(f"Warning: bad gradient at step {step}. Skipping weight update to protect model weights.")
            opt.zero_grad(set_to_none=True)
        else:
            clip_grad_norm_(trainable, 1.0)
            opt.step()
            opt.zero_grad(set_to_none=True)

        if (step + 1) % 25 == 0 or step == steps - 1:
            avg_loss = sum(losses)/len(losses) if losses else 0.0
            print(f"[{setup_name}] Step {step+1}/{steps} - Loss: {avg_loss:.4f}")

    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3)

    adapter_dir = f"work/adapters/{eval_name}"
    os.makedirs(adapter_dir, exist_ok=True)
    peft_model.save_pretrained(adapter_dir)

    run_meta = {
        "setup": setup_name,
        "trainable_params": sum(p.numel() for p in trainable),
        "train_minutes": round(train_minutes, 2),
        "peak_vram_gb": round(peak_vram_gb, 3)
    }
    with open(os.path.join(adapter_dir, "run.json"), "w") as f:
        json.dump(run_meta, f, indent=2)
    print(f"Saved run metadata for {setup_name}:", run_meta)

    # Inference
    print(f"Running test inference on {len(test_rows)} images for {setup_name}...")
    peft_model.eval()
    pred_path = f"work/{eval_name}.jsonl"
    os.makedirs("work", exist_ok=True)

    with open(pred_path, "w", encoding="utf-8") as out_f:
        for row in tqdm(test_rows, desc=f"Inference {setup_name}"):
            messages = make_messages(row)
            inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")
            inputs = {k: v.to(peft_model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
            with torch.inference_mode():
                generated = peft_model.generate(**inputs, max_new_tokens=128, do_sample=False)
            prompt_len = inputs["input_ids"].shape[1]
            raw = processor.batch_decode(generated[:, prompt_len:], skip_special_tokens=True)[0]
            out_f.write(json.dumps({"image_id": row["image_id"], "raw": raw}, ensure_ascii=False) + "\n")

    # Scoring
    score_cmd = [
        "python", "scripts/score.py",
        "--pred", pred_path,
        "--name", setup_name,
        "--out", str(evidence_path),
        "--meta", f"trainable_params={run_meta['trainable_params']}",
        "--meta", f"peak_vram_gb={run_meta['peak_vram_gb']}",
        "--meta", f"train_minutes={run_meta['train_minutes']}"
    ]
    subprocess.run(score_cmd, check=True)
    print(f"Score completed for {setup_name}.")

    del peft_model, base_model, opt, trainable
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# 5. Execute Experiments E2, E3, E4
run_experiment("E2", rank=16, base_lr=1e-5)
run_experiment("E3", rank=16, base_lr=1e-5)
run_experiment("E4_r64", rank=64, base_lr=1e-4)


In [ ]:
# 6. Generate 6 Visual Figures for Report Q9
from PIL import ImageDraw
from scripts.score import parse_output

fig_dir = Path("report/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

pred_file = "work/e2.jsonl" if Path("work/e2.jsonl").exists() else "work/e1.jsonl"
preds_map = {r["image_id"]: r["raw"] for r in read_jsonl(pred_file)}

def draw_sample(row, pred_raw, save_path, desc=""):
    img_path = row["image_path"] if os.path.exists(row["image_path"]) else os.path.join("data", row["image_path"])
    im = Image.open(img_path).convert("RGB")
    w, h = im.size
    draw = ImageDraw.Draw(im)
    # GT in Green
    for obj in row["objects"]:
        b = yolo_to_box(obj["label"], obj["yolo"])["bbox_2d"]
        x1, y1, x2, y2 = b[0]*w/1000.0, b[1]*h/1000.0, b[2]*w/1000.0, b[3]*h/1000.0
        draw.rectangle([x1, y1, x2, y2], outline=(0, 255, 0), width=3)
        draw.text((x1, max(0, y1-12)), f"GT: {obj['label']}", fill=(0, 255, 0))
    # Pred in Red
    for p in (parse_output(pred_raw) or []):
        b = p["bbox"]
        x1, y1, x2, y2 = b[0]*w/1000.0, b[1]*h/1000.0, b[2]*w/1000.0, b[3]*h/1000.0
        draw.rectangle([x1, y1, x2, y2], outline=(255, 60, 60), width=2)
        draw.text((x1, min(h-12, y2+2)), f"PRED: {p['label']}", fill=(255, 60, 60))
    im.save(save_path)
    print(f"Saved {save_path.name} ({desc})")

titles = ["pothole_1", "crack_1", "crack_2", "manhole_1", "multi_damage", "complex_road"]
for i, name in enumerate(titles):
    idx = (i * 23) % len(test_rows)
    row = test_rows[idx]
    draw_sample(row, preds_map.get(row["image_id"], ""), fig_dir / f"fig_{name}.jpg", desc=name)

print("6 figures generated successfully.")


In [ ]:
# 7. Check Submission & Export Evidence
!python scripts/check_submission.py

if IS_KAGGLE:
    out_ev = Path("/kaggle/working/report/evidence")
    out_fg = Path("/kaggle/working/report/figures")
    out_ev.mkdir(parents=True, exist_ok=True)
    out_fg.mkdir(parents=True, exist_ok=True)
    for f in Path("report/evidence").glob("*.json"):
        shutil.copy2(f, out_ev / f.name)
    for f in Path("report/figures").glob("*.jpg"):
        shutil.copy2(f, out_fg / f.name)
    print("\n--- Exported to /kaggle/working/report/evidence: ---")
    for f in out_ev.iterdir(): print(f"  {f.name} ({f.stat().st_size} bytes)")
    print("\n--- Exported to /kaggle/working/report/figures: ---")
    for f in out_fg.iterdir(): print(f"  {f.name} ({f.stat().st_size} bytes)")
